# Performance Plotting

Notebook for the current `performance-sionna-tcp` output layout. It reads the `tc*/throughput_results_*`, `snr_results_*`, `position_results_*`, and `computationtime_*` CSV files, and treats each configured result folder as one comparable server/raytracer setting.

Use `0.1` for a 0.1 ms coherence time if you set coherence times manually.

In [ ]:
%matplotlib inline

from pathlib import Path
import sys
import matplotlib.pyplot as plt

cwd = Path.cwd()
EXAMPLES_DIR = cwd if (cwd / "performance-sionna-tcp.cc").exists() else cwd / "examples"
if not EXAMPLES_DIR.exists():
    EXAMPLES_DIR = cwd
if str(EXAMPLES_DIR) not in sys.path:
    sys.path.insert(0, str(EXAMPLES_DIR))

from performance_plotting import (
    default_apartment_server_datasets,
    load_performance_data,
    plot_metric_boxplot,
    plot_metric_vs_coherence,
    plot_positions,
    plot_server_setting_comparison,
    plot_time_series,
)

# Save switch. Keep this False while exploring; set True to write figures to FIGURE_DIR.
SAVE_PLOTS = False
FIGURE_DIR = EXAMPLES_DIR / "plots"

# None means auto-detect everything available. Example: [0.1, 1, 10, 100, 1000]
COHERENCE_TIMES = None
SEEDS = None

THROUGHPUT_INTERVAL_MS = 100
DROP_ZERO_THROUGHPUT = True
SINGLE_DATASET = "Current"
SINGLE_SEED = 1
SINGLE_COHERENCE_MS = 1
SNR_STA_NODE_IDS = None
AP_POSITION = (-1.0, 0.5)

CURRENT_DATASETS = {
    "Current": EXAMPLES_DIR / "results",
}

SERVER_DATASETS = default_apartment_server_datasets(EXAMPLES_DIR / "results_Apartment")
SERVER_DATASETS

## Load Data

In [ ]:
current_data = load_performance_data(
    CURRENT_DATASETS,
    coherence_times=COHERENCE_TIMES,
    seeds=SEEDS,
)
current_data.summary()

In [ ]:
server_data = load_performance_data(
    SERVER_DATASETS,
    coherence_times=COHERENCE_TIMES,
    seeds=SEEDS,
)
server_data.summary()

## Coherence Sweeps

In [ ]:
try:
    plot_metric_vs_coherence(
        current_data,
        metric="throughput",
        interval_ms=THROUGHPUT_INTERVAL_MS,
        drop_zero_throughput=DROP_ZERO_THROUGHPUT,
        save=SAVE_PLOTS,
        output_dir=FIGURE_DIR,
        filename="throughput_vs_coherence_current.png",
    )
    plt.show()
except ValueError as exc:
    print(f"[skip] Throughput coherence sweep: {exc}")

try:
    plot_metric_vs_coherence(
        current_data,
        metric="snr",
        interval_ms=THROUGHPUT_INTERVAL_MS,
        drop_zero_throughput=DROP_ZERO_THROUGHPUT,
        sta_node_ids=SNR_STA_NODE_IDS,
        save=SAVE_PLOTS,
        output_dir=FIGURE_DIR,
        filename="snr_vs_coherence_current.png",
    )
    plt.show()
except ValueError as exc:
    print(f"[skip] SNR coherence sweep: {exc}")

## Single Seed Time Series

In [ ]:
try:
    plot_time_series(
        current_data,
        metric="throughput",
        dataset=SINGLE_DATASET,
        coherence_ms=SINGLE_COHERENCE_MS,
        seed=SINGLE_SEED,
        interval_ms=THROUGHPUT_INTERVAL_MS,
        save=SAVE_PLOTS,
        output_dir=FIGURE_DIR,
        filename=f"throughput_time_seed{SINGLE_SEED}_tc{SINGLE_COHERENCE_MS}.png",
    )
    plt.show()
except ValueError as exc:
    print(f"[skip] Throughput time series: {exc}")

try:
    plot_time_series(
        current_data,
        metric="snr",
        dataset=SINGLE_DATASET,
        coherence_ms=SINGLE_COHERENCE_MS,
        seed=SINGLE_SEED,
        sta_node_ids=SNR_STA_NODE_IDS,
        save=SAVE_PLOTS,
        output_dir=FIGURE_DIR,
        filename=f"snr_time_seed{SINGLE_SEED}_tc{SINGLE_COHERENCE_MS}.png",
    )
    plt.show()
except ValueError as exc:
    print(f"[skip] SNR time series: {exc}")

## Positions

In [ ]:
try:
    plot_positions(
        current_data,
        dataset=SINGLE_DATASET,
        coherence_times=COHERENCE_TIMES or [SINGLE_COHERENCE_MS],
        seed=SINGLE_SEED,
        sta_node_ids=SNR_STA_NODE_IDS,
        ap_position=AP_POSITION,
        save=SAVE_PLOTS,
        output_dir=FIGURE_DIR,
        filename=f"positions_seed{SINGLE_SEED}.png",
    )
    plt.show()
except ValueError as exc:
    print(f"[skip] Position plot: {exc}")

## Boxplots

In [ ]:
boxplot_data = server_data if not server_data.throughput.empty else current_data

for metric in ["computation_time", "throughput", "snr"]:
    try:
        plot_metric_boxplot(
            boxplot_data,
            metric=metric,
            interval_ms=THROUGHPUT_INTERVAL_MS,
            drop_zero_throughput=DROP_ZERO_THROUGHPUT,
            sta_node_ids=SNR_STA_NODE_IDS,
            save=SAVE_PLOTS,
            output_dir=FIGURE_DIR,
            filename=f"{metric}_boxplot.png",
        )
        plt.show()
    except ValueError as exc:
        print(f"[skip] {metric} boxplot: {exc}")

## Server/Raytracer Settings

In [ ]:
for metric in ["throughput", "snr"]:
    try:
        plot_server_setting_comparison(
            server_data,
            metric=metric,
            interval_ms=THROUGHPUT_INTERVAL_MS,
            drop_zero_throughput=DROP_ZERO_THROUGHPUT,
            coherence_times=COHERENCE_TIMES,
            sta_node_ids=SNR_STA_NODE_IDS,
            save=SAVE_PLOTS,
            output_dir=FIGURE_DIR,
            filename=f"{metric}_server_settings.png",
        )
        plt.show()
    except ValueError as exc:
        print(f"[skip] {metric} server setting comparison: {exc}")

## Add A Comparison

In [ ]:
def quick_compare(label_to_path, metric="throughput", coherence_times=None, seeds=None, interval_ms=THROUGHPUT_INTERVAL_MS):
    """Load any result folders and draw the two comparison plots used most often."""
    comparison = load_performance_data(label_to_path, coherence_times=coherence_times, seeds=seeds)
    display(comparison.summary())

    plot_metric_vs_coherence(
        comparison,
        metric=metric,
        interval_ms=interval_ms,
        drop_zero_throughput=DROP_ZERO_THROUGHPUT,
        save=SAVE_PLOTS,
        output_dir=FIGURE_DIR,
        filename=f"{metric}_quick_vs_coherence.png",
    )
    plt.show()

    plot_server_setting_comparison(
        comparison,
        metric=metric,
        interval_ms=interval_ms,
        drop_zero_throughput=DROP_ZERO_THROUGHPUT,
        coherence_times=coherence_times,
        save=SAVE_PLOTS,
        output_dir=FIGURE_DIR,
        filename=f"{metric}_quick_settings.png",
    )
    plt.show()
    return comparison

# Example:
# my_data = quick_compare({
#     "Full": EXAMPLES_DIR / "results_Apartment" / "results_FullNew",
#     "No reflection": EXAMPLES_DIR / "results_Apartment" / "results_noReflecNew",
# }, metric="snr", coherence_times=[0.1, 1, 10], seeds=range(1, 6))